# SFT Adapter Eval

Simple before/after sanity check on the SFT LoRA adapter.

- **Base**: `mistralai/Mistral-7B-v0.1` in bf16 (no quantization needed for inference on 4090)
- **Adapter**: `checkpoints/sft-zephyr-lora/checkpoint-17205` (1 epoch UltraChat, trained on A100)

In [ ]:
%pip install -q peft transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL   = "mistralai/Mistral-7B-v0.1"
ADAPTER_PATH = "../checkpoints/sft-zephyr-lora/checkpoint-17205"

TEST_PROMPTS = [
    [{"role": "user", "content": "Explain the difference between supervised learning and reinforcement learning in 3 sentences."}],
    [{"role": "user", "content": "Write a short haiku about gradient descent."}],
    [{"role": "user", "content": "What is 17 × 23? Show your work step by step."}],
]

print("CUDA:", torch.cuda.is_available())
print("GPU: ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## 1. Load Model

Mistral-7B in bf16 = ~14 GB VRAM — fits on the 4090 with room to spare. No quantization needed for inference.

In [ ]:
# Load tokenizer from checkpoint — it has the Zephyr chat template baked in
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"VRAM:   {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
@torch.inference_mode()
def generate(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)

## 2. Base Model Predictions

In [ ]:
print("=" * 60)
print("BASE MODEL")
print("=" * 60)

base_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    base_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

## 3. Load SFT Adapter

In [ ]:
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print("Adapter loaded.")

## 4. Post-SFT Predictions

In [ ]:
print("=" * 60)
print("POST-SFT MODEL")
print("=" * 60)

sft_outputs = []
for msgs in TEST_PROMPTS:
    out = generate(msgs)
    sft_outputs.append(out)
    print(f"\nQ: {msgs[0]['content']}")
    print(f"A: {out}")
    print("-" * 60)

## 5. Side-by-Side Comparison

In [ ]:
for i, msgs in enumerate(TEST_PROMPTS):
    print("=" * 60)
    print(f"Q: {msgs[0]['content']}")
    print(f"\n[BASE]\n{base_outputs[i]}")
    print(f"\n[SFT]\n{sft_outputs[i]}")
    print()